# txt2reward — LLM-as-Reward for Highway-RL

PPO on `highway-v0` with a Groq LLM (llama-3.3-70b) as the reward signal.

> **Make sure the GPU runtime is enabled** — Runtime → Change runtime type → T4 GPU

## 1. Install dependencies

In [ ]:
%%capture
!pip install gymnasium highway-env stable-baselines3 groq numpy

## 2. Mount Google Drive

All checkpoints and the reward cache are saved here automatically.
After a disconnect, re-run from **Step 6 (Resume)** — no work is lost.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/txt2reward'

import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive ready:', DRIVE_DIR)

## 3. Set GROQ_API_KEY

Get your key at [console.groq.com](https://console.groq.com).

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "gsk_xxxxxxxxxxxxxxxx"  # <- paste your key here

from groq import Groq
_ = Groq(api_key=os.environ["GROQ_API_KEY"])
print("Groq client OK")

## 4. Clone the repository

In [ ]:
!git clone https://github.com/A-Nematkhah/txt2reward.git
%cd txt2reward

## 5. Restore cache from Drive (if available)

If the cache was already built in a previous session, copy it back.
Skip this cell on the very first run.

In [ ]:
import shutil, os

drive_cache = f'{DRIVE_DIR}/reward_cache.pkl'
if os.path.exists(drive_cache):
    shutil.copy(drive_cache, 'reward_cache.pkl')
    print('Cache restored from Drive')
else:
    print('No cache on Drive yet — will build from scratch in next step')

## 6. Pre-fill the reward cache

Queries Groq for all **192 possible states** (~8 minutes on free tier).
The cache is saved to Drive immediately after — safe to disconnect after this cell.

In [ ]:
from llm_judge import prefill_cache_sync
import shutil

prefill_cache_sync(verbose=True)

# Back up to Drive right away
shutil.copy('reward_cache.pkl', f'{DRIVE_DIR}/reward_cache.pkl')
print('Cache backed up to Drive')

## 7. Sanity-check the LLM judge

In [ ]:
from llm_judge import judge_state, reward_cache

test_cases = [
    (30.0, 2, 25.0, "expected: positive"),
    (10.0, 0,  3.0, "expected: negative"),
    (40.0, 3,  8.0, "expected: negative"),
]
for speed, lane, dist, label in test_cases:
    r = judge_state(speed, lane, dist)
    print(f"speed={speed:4.0f}  lane={lane}  dist={dist:4.0f}m  reward={r:+.3f}  ({label})")

print("Cache size:", len(reward_cache), "entries")

## 8. Train

Checkpoints are saved every **10,000 steps** locally and synced to Drive.
If Colab disconnects, jump to **Step 9 (Resume)**.

In [ ]:
!python train.py --timesteps 100000 --n-envs 4 --no-precompute --drive-dir $DRIVE_DIR --checkpoint-freq 10000

## 9. Resume after disconnect

Re-run cells 1–5 first, then run this cell.
It finds the latest checkpoint on Drive and continues from there.

In [ ]:
import os, glob, shutil

checkpoints = sorted(
    glob.glob(f'{DRIVE_DIR}/ppo_highway_*.zip'),
    key=lambda p: int(p.split('_steps')[0].split('_')[-1])
)

if not checkpoints:
    print("No checkpoints found on Drive — start a fresh run (Step 8)")
else:
    latest = checkpoints[-1]
    print("Resuming from:", latest)
    shutil.copy(latest, os.path.basename(latest))
    shutil.copy(f'{DRIVE_DIR}/reward_cache.pkl', 'reward_cache.pkl')
    completed = int(latest.split('_steps')[0].split('_')[-1])
    remaining = max(0, 100_000 - completed)
    print(f"Steps completed: {completed:,} — remaining: {remaining:,}")
    resume_path = os.path.basename(latest)
    cmd = (
        f'python train.py'
        f' --timesteps {remaining}'
        f' --n-envs 4'
        f' --no-precompute'
        f' --resume {resume_path}'
        f' --drive-dir {DRIVE_DIR}'
        f' --checkpoint-freq 10000'
    )
    os.system(cmd)


## 10. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir tb_logs

## 11. Evaluate the trained model

In [ ]:
import gymnasium as gym
import highway_env
from stable_baselines3 import PPO
from train import make_env

env   = make_env(rank=0, llm_interval=50)()
model = PPO.load("ppo_highway_qwen_reward", env=env)

n_episodes = 5
rewards    = []
for ep in range(n_episodes):
    obs, _ = env.reset()
    done, ep_r = False, 0.0
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, r, terminated, truncated, _ = env.step(action)
        ep_r += r
        done  = terminated or truncated
    rewards.append(ep_r)
    print(f"Episode {ep + 1:2d}: total reward = {ep_r:.2f}")

print("Mean reward over", n_episodes, "episodes:", round(sum(rewards) / n_episodes, 2))
env.close()